# PyTorch tutorial for Milan workshop
code from Gemini with prompt "Write a python script to classify the MNIST dataset"

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

In [2]:
# 1. Setup Device (Use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# 2. Define the CNN Architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [4]:
# 3. Data Loading & Preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=64, shuffle=True)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=False, transform=transform),
    batch_size=1000)

100%|██████████| 9.91M/9.91M [00:01<00:00, 7.05MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 270kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 703kB/s]


In [5]:
# 4. Initialize Model, Optimizer, and Loss
model = Net().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=1.0)

In [6]:
# 5. Training Loop
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}")

In [7]:
# 6. Evaluation
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    print(f"\nTest set: Average loss: {test_loss/len(test_loader.dataset):.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({100. * correct / len(test_loader.dataset):.2f}%)\n")

In [8]:
# Run the training for 1 epoch
if __name__ == '__main__':
    train(model, device, train_loader, optimizer, 1)
    test(model, device, test_loader)

Train Epoch: 1 [0/60000] Loss: 2.319513
Train Epoch: 1 [6400/60000] Loss: 0.063531
Train Epoch: 1 [12800/60000] Loss: 0.065041
Train Epoch: 1 [19200/60000] Loss: 0.038905
Train Epoch: 1 [25600/60000] Loss: 0.071349
Train Epoch: 1 [32000/60000] Loss: 0.130733
Train Epoch: 1 [38400/60000] Loss: 0.193958
Train Epoch: 1 [44800/60000] Loss: 0.145396
Train Epoch: 1 [51200/60000] Loss: 0.001412
Train Epoch: 1 [57600/60000] Loss: 0.013661

Test set: Average loss: 0.0413, Accuracy: 9867/10000 (98.67%)

